In [ ]:
# -*- coding: utf-8 -*-
"""Complete Workflow: NER (Multi‑word BIO) + Sentiment Analysis with Cross‑Validation"""

# ============================
# A. Install Required Libraries
# ============================
!pip install datasets tokenizers seqeval -q
!pip install transformers==4.17
!pip install torch
!pip install sentencepiece
!pip install evaluate
!pip install transformers datasets evaluate

# ============================
# B. Import Libraries
# ============================
import json
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, auc
)
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForTokenClassification,
    AutoModelForSequenceClassification, TrainingArguments, Trainer,
    DataCollatorForTokenClassification, DataCollatorWithPadding
)
import evaluate
import torch

import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# -*- coding: utf-8 -*-
"""Final Workflow: NER (Multi‑word BIO) + Sentiment Analysis (No CV)"""

# ============================
# A. Install & Import
# ============================
!pip install datasets tokenizers seqeval -q
!pip install transformers==4.17
!pip install torch
!pip install transformers datasets evaluate

import os
os.environ["WANDB_DISABLED"] = "true"

import json
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, auc
)
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForTokenClassification,
    AutoModelForSequenceClassification, TrainingArguments, Trainer,
    DataCollatorForTokenClassification, DataCollatorWithPadding
)
import evaluate

from google.colab import drive
drive.mount('/content/drive')

# ============================
# B. Load Data
# ============================
TRAIN_CSV = '/content/Gemini 3.5-Flash_Extended_Synthetic_Dataset.csv'
TEST_CSV  = '/content/Test.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f"Training samples: {len(df_train)}")
print(f"Test samples: {len(df_test)}")

# ============================
# C. Parse BIO Tags -> Tokens & NER Labels
# ============================
def parse_bio_tags(bio_text):
    tokens, labels = [], []
    for part in bio_text.split():
        match = re.match(r'^(.*?)(?:[,\s।]*)-(B|I)$', part)
        if match:
            token, tag = match.group(1), match.group(2)
            labels.append('B-Product' if tag == 'B' else 'I-Product')
        else:
            token = part
            labels.append('O')
        tokens.append(token)
    return tokens, labels

df_train['tokens'] = df_train['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
df_train['ner_tags'] = df_train['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])

if 'bio_tagged_review' in df_test.columns:
    df_test['tokens'] = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
    df_test['ner_tags'] = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])
else:
    df_test['tokens'] = df_test['review'].apply(lambda x: x.split())

# ============================
# D. Label Mappings
# ============================
label_list = ['O', 'B-Product', 'I-Product']
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}
num_labels = len(label_list)

# ============================
# E. Split Training Data (80/20)
# ============================
X = df_train[['id', 'review', 'tokens', 'ner_tags']]
y = df_train['sentiment']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}")
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)

# ============================
# F. NER Tokenizer & Alignment
# ============================
tokenizer_ner = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_and_align_labels(examples, tokenizer, label2id):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding=False
    )
    all_labels = []
    for i, label_seq in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_seq[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

def create_ner_dataset(df, tokenizer, label2id):
    tokenized = tokenize_and_align_labels(
        {'tokens': df['tokens'].tolist(), 'ner_tags': df['ner_tags'].tolist()},
        tokenizer,
        label2id
    )
    return Dataset.from_dict({
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': tokenized['labels'],
        'tokens': df['tokens'].tolist(),
        'ner_tags': df['ner_tags'].tolist(),
        'id': df['id'].tolist() if 'id' in df else list(range(len(df)))
    })

train_dataset_ner = create_ner_dataset(X_train, tokenizer_ner, label2id)
val_dataset_ner   = create_ner_dataset(X_val, tokenizer_ner, label2id)

if 'tokens' in df_test.columns and 'ner_tags' in df_test.columns:
    test_dataset_ner = create_ner_dataset(df_test, tokenizer_ner, label2id)
else:
    test_dataset_ner = None

# ============================
# G. Train Final NER Model (no CV)
# ============================
print("\n=== Training NER model on full training data ===")
full_train = pd.concat([X_train, X_val], ignore_index=True)
full_train['sentiment'] = pd.concat([y_train, y_val], ignore_index=True)
full_train_ds = create_ner_dataset(full_train, tokenizer_ner, label2id)

config = AutoConfig.from_pretrained("xlm-roberta-base")
config.num_labels = num_labels
config.id2label = id2label
config.label2id = label2id
model_ner = AutoModelForTokenClassification.from_config(config)

args_ner = TrainingArguments(
    output_dir="ner_final",
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
)

data_collator_ner = DataCollatorForTokenClassification(tokenizer_ner)
metric_ner = evaluate.load("seqeval")

def compute_metrics_ner(eval_preds):
    pred_logits, labels = eval_preds
    pred_logits = np.argmax(pred_logits, axis=2)
    predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(pred_logits, labels)
    ]
    references = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(pred_logits, labels)
    ]
    results = metric_ner.compute(predictions=predictions, references=references)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

trainer_ner = Trainer(
    model=model_ner,
    args=args_ner,
    train_dataset=full_train_ds,
    eval_dataset=val_dataset_ner,
    data_collator=data_collator_ner,
    tokenizer=tokenizer_ner,
    compute_metrics=compute_metrics_ner
)

trainer_ner.train()

if test_dataset_ner is not None:
    test_results_ner = trainer_ner.evaluate(test_dataset_ner)
    print("\n=== NER Test Results ===")
    print(test_results_ner)

model_ner.save_pretrained("ner_model_final")
tokenizer_ner.save_pretrained("ner_tokenizer")

# ============================
# H. Sentiment Analysis (no CV)
# ============================
sa_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_sa(reviews, tokenizer):
    return tokenizer(reviews, truncation=True, padding=False)

X_train_sa = X_train['review'].tolist()
y_train_sa = y_train.tolist()
X_val_sa = X_val['review'].tolist()
y_val_sa = y_val.tolist()
full_train_sa = full_train['review'].tolist()
full_train_sa_y = full_train['sentiment'].tolist()

train_enc_sa = tokenize_sa(X_train_sa, sa_tokenizer)
val_enc_sa = tokenize_sa(X_val_sa, sa_tokenizer)
full_train_enc_sa = tokenize_sa(full_train_sa, sa_tokenizer)

train_sa_dataset = Dataset.from_dict({
    'input_ids': train_enc_sa['input_ids'],
    'attention_mask': train_enc_sa['attention_mask'],
    'labels': y_train_sa
})
val_sa_dataset = Dataset.from_dict({
    'input_ids': val_enc_sa['input_ids'],
    'attention_mask': val_enc_sa['attention_mask'],
    'labels': y_val_sa
})
full_train_sa_dataset = Dataset.from_dict({
    'input_ids': full_train_enc_sa['input_ids'],
    'attention_mask': full_train_enc_sa['attention_mask'],
    'labels': full_train_sa_y
})

print("\n=== Training final SA model on full training data ===")
model_sa = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3,
    id2label={0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"},
    label2id={"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
)

args_sa = TrainingArguments(
    output_dir="sa_final",
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
)

data_collator_sa = DataCollatorWithPadding(sa_tokenizer)

def compute_metrics_sa_final(eval_pred):
    preds = eval_pred.predictions.argmax(-1)
    labels = eval_pred.label_ids
    return {
        'precision': precision_score(labels, preds, average='weighted'),
        'recall': recall_score(labels, preds, average='weighted'),
        'f1': f1_score(labels, preds, average='weighted'),
        'accuracy': accuracy_score(labels, preds)
    }

trainer_sa = Trainer(
    model=model_sa,
    args=args_sa,
    train_dataset=full_train_sa_dataset,
    eval_dataset=val_sa_dataset,
    tokenizer=sa_tokenizer,
    data_collator=data_collator_sa,
    compute_metrics=compute_metrics_sa_final
)

trainer_sa.train()

# ============================
# I. Evaluate SA on Test Set
# ============================
test_reviews = df_test['review'].tolist()
test_enc_sa = tokenize_sa(test_reviews, sa_tokenizer)
if 'sentiment' in df_test.columns:
    test_labels = df_test['sentiment'].tolist()
else:
    test_labels = [0] * len(df_test)
    print("⚠️ No sentiment column in test set – using dummy labels.")

test_sa_dataset = Dataset.from_dict({
    'input_ids': test_enc_sa['input_ids'],
    'attention_mask': test_enc_sa['attention_mask'],
    'labels': test_labels
})

test_results_sa = trainer_sa.evaluate(test_sa_dataset)
print("\n=== SA Test Results ===")
print(test_results_sa)

model_sa.save_pretrained("sa_model_final")
sa_tokenizer.save_pretrained("sa_tokenizer")

# ============================
# J. Inference Example (with device fix)
# ============================
from transformers import pipeline

# NER pipeline
ner_pipeline = pipeline(
    "ner",
    model="ner_model_final",
    tokenizer="ner_tokenizer",
    aggregation_strategy="simple"
)

sample_review = "ফেস ওয়াশ টা অনেক সুন্দর, অসংখ্য ধন্যবাদ ডেলিভারি খুব দ্রুত দেয়ার জন্য।"
print("\nSample review:", sample_review)
ner_results = ner_pipeline(sample_review)
print("NER results:", ner_results)

# Sentiment inference with device handling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sa.to(device)

sample_review_sa = [sample_review]
enc = sa_tokenizer(sample_review_sa, padding=True, truncation=True, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}  # move inputs to same device

with torch.no_grad():
    logits = model_sa(**enc).logits
    pred = logits.argmax(dim=1).item()

sentiment_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
print(f"Sentiment: {sentiment_map[pred]}")

print("\nAll tasks completed successfully!")

Mounted at /content/drive
Training samples: 3000
Test samples: 30
Train size: 2400, Validation size: 600


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



=== Training NER model on full training data ===


Using the `WAND_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The following columns in the training set  don't have a corresponding argument in `XLMRobertaForTokenClassification.forward` and have been ignored: tokens, ner_tags, id. If tokens, ner_tags, id are not expected by `XLMRobertaForTokenClassification.forward`,  you can safely ignore thi

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.089800,0.049908,0.872593,0.887048,0.879761,0.983136
2,0.027600,0.018973,0.945166,0.986446,0.965365,0.993619
3,0.012300,0.009010,0.995420,0.981928,0.988628,0.997873


...

***** Running Evaluation *****
  Num examples = 600
  Batch size = 16
Saving model checkpoint to sa_final/checkpoint-188
Configuration saved in sa_final/checkpoint-188/config.json
Model weights saved in sa_final/checkpoint-188/pytorch_model.bin
tokenizer config file saved in sa_final/checkpoint-188/tokenizer_config.json
Special tokens file saved in sa_final/checkpoint-188/special_tokens_map.json
***** Running Evaluation *****
  Num examples = 600
  Batch size = 16
Saving model checkpoint to sa_final/checkpoint-376
Configuration saved in sa_final/checkpoint-376/config.json
Model weights saved in sa_final/checkpoint-376/pytorch_model.bin
tokenizer config file saved in sa_final/checkpoint-376/tokenizer_config.json
Special tokens file saved in sa_final/checkpoint-376/special_tokens_map.json
***** Running Evaluation *****
  Num examples = 600
  Batch size = 16
Saving model checkpoint to sa_final/checkpoint-564
Configuration saved in sa_final/checkpoint-564/config.json
Model weights saved in

Configuration saved in sa_model_final/config.json



=== SA Test Results ===
{'eval_loss': 1.3486202955245972, 'eval_precision': 0.7583333333333333, 'eval_recall': 0.7, 'eval_f1': 0.7094054580896686, 'eval_accuracy': 0.7, 'eval_runtime': 0.1166, 'eval_samples_per_second': 257.261, 'eval_steps_per_second': 17.151, 'epoch': 3.0}


Model weights saved in sa_model_final/pytorch_model.bin
tokenizer config file saved in sa_tokenizer/tokenizer_config.json
Special tokens file saved in sa_tokenizer/special_tokens_map.json
loading configuration file ner_model_final/config.json
Model config XLMRobertaConfig {
  "_name_or_path": "ner_model_final",
  "architectures": [
    "XLMRobertaForTokenClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-Product",
    "2": "I-Product"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "B-Product": 1,
    "I-Product": 2,
    "O": 0
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absol


Sample review: ফেস ওয়াশ টা অনেক সুন্দর, অসংখ্য ধন্যবাদ ডেলিভারি খুব দ্রুত দেয়ার জন্য।
NER results: [{'entity_group': 'Product', 'score': np.float32(0.9999869), 'word': 'ফ', 'start': 0, 'end': 1}, {'entity_group': 'Product', 'score': np.float32(0.99951625), 'word': 'ওয়া', 'start': 4, 'end': 8}]
Sentiment: Positive

All tasks completed successfully!


In [ ]:

# ============================
# B. Load Data
# ============================
TRAIN_CSV = '/content/Gemini 3.5-Flash_Extended_Synthetic_Dataset.csv'
TEST_CSV  = '/content/Test.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f"Training samples: {len(df_train)}")
print(f"Test samples: {len(df_test)}")

# ============================
# C. Parse BIO Tags -> Tokens & NER Labels
# ============================
def parse_bio_tags(bio_text):
    tokens, labels = [], []
    for part in bio_text.split():
        match = re.match(r'^(.*?)(?:[,\s।]*)-(B|I)$', part)
        if match:
            token, tag = match.group(1), match.group(2)
            labels.append('B-Product' if tag == 'B' else 'I-Product')
        else:
            token = part
            labels.append('O')
        tokens.append(token)
    return tokens, labels

df_train['tokens'] = df_train['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
df_train['ner_tags'] = df_train['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])

if 'bio_tagged_review' in df_test.columns:
    df_test['tokens'] = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
    df_test['ner_tags'] = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])
else:
    df_test['tokens'] = df_test['review'].apply(lambda x: x.split())

# ============================
# D. Label Mappings
# ============================
label_list = ['O', 'B-Product', 'I-Product']
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}
num_labels = len(label_list)

# ============================
# E. Split Training Data (80/20)
# ============================
X = df_train[['id', 'review', 'tokens', 'ner_tags']]
y = df_train['sentiment']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}")
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)

# ============================
# F. NER Tokenizer & Alignment
# ============================
tokenizer_ner = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_and_align_labels(examples, tokenizer, label2id):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding=False
    )
    all_labels = []
    for i, label_seq in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_seq[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

def create_ner_dataset(df, tokenizer, label2id):
    tokenized = tokenize_and_align_labels(
        {'tokens': df['tokens'].tolist(), 'ner_tags': df['ner_tags'].tolist()},
        tokenizer,
        label2id
    )
    return Dataset.from_dict({
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': tokenized['labels'],
        'tokens': df['tokens'].tolist(),
        'ner_tags': df['ner_tags'].tolist(),
        'id': df['id'].tolist() if 'id' in df else list(range(len(df)))
    })

train_dataset_ner = create_ner_dataset(X_train, tokenizer_ner, label2id)
val_dataset_ner   = create_ner_dataset(X_val, tokenizer_ner, label2id)

if 'tokens' in df_test.columns and 'ner_tags' in df_test.columns:
    test_dataset_ner = create_ner_dataset(df_test, tokenizer_ner, label2id)
else:
    test_dataset_ner = None

# ============================
# G. Train Final NER Model (no CV)
# ============================
print("\n=== Training NER model on full training data ===")
full_train = pd.concat([X_train, X_val], ignore_index=True)
full_train['sentiment'] = pd.concat([y_train, y_val], ignore_index=True)
full_train_ds = create_ner_dataset(full_train, tokenizer_ner, label2id)

config = AutoConfig.from_pretrained("xlm-roberta-base")
config.num_labels = num_labels
config.id2label = id2label
config.label2id = label2id
model_ner = AutoModelForTokenClassification.from_config(config)

args_ner = TrainingArguments(
    output_dir="ner_final",
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
)

data_collator_ner = DataCollatorForTokenClassification(tokenizer_ner)
metric_ner = evaluate.load("seqeval")

def compute_metrics_ner(eval_preds):
    pred_logits, labels = eval_preds
    pred_logits = np.argmax(pred_logits, axis=2)
    predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(pred_logits, labels)
    ]
    references = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(pred_logits, labels)
    ]
    results = metric_ner.compute(predictions=predictions, references=references)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

trainer_ner = Trainer(
    model=model_ner,
    args=args_ner,
    train_dataset=full_train_ds,
    eval_dataset=val_dataset_ner,
    data_collator=data_collator_ner,
    tokenizer=tokenizer_ner,
    compute_metrics=compute_metrics_ner
)

trainer_ner.train()

if test_dataset_ner is not None:
    test_results_ner = trainer_ner.evaluate(test_dataset_ner)
    print("\n=== NER Test Results ===")
    print(test_results_ner)

model_ner.save_pretrained("ner_model_final")
tokenizer_ner.save_pretrained("ner_tokenizer")

# ============================
# H. Sentiment Analysis (no CV)
# ============================
sa_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_sa(reviews, tokenizer):
    return tokenizer(reviews, truncation=True, padding=False)

X_train_sa = X_train['review'].tolist()
y_train_sa = y_train.tolist()
X_val_sa = X_val['review'].tolist()
y_val_sa = y_val.tolist()
full_train_sa = full_train['review'].tolist()
full_train_sa_y = full_train['sentiment'].tolist()

train_enc_sa = tokenize_sa(X_train_sa, sa_tokenizer)
val_enc_sa = tokenize_sa(X_val_sa, sa_tokenizer)
full_train_enc_sa = tokenize_sa(full_train_sa, sa_tokenizer)

train_sa_dataset = Dataset.from_dict({
    'input_ids': train_enc_sa['input_ids'],
    'attention_mask': train_enc_sa['attention_mask'],
    'labels': y_train_sa
})
val_sa_dataset = Dataset.from_dict({
    'input_ids': val_enc_sa['input_ids'],
    'attention_mask': val_enc_sa['attention_mask'],
    'labels': y_val_sa
})
full_train_sa_dataset = Dataset.from_dict({
    'input_ids': full_train_enc_sa['input_ids'],
    'attention_mask': full_train_enc_sa['attention_mask'],
    'labels': full_train_sa_y
})

print("\n=== Training final SA model on full training data ===")
model_sa = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3,
    id2label={0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"},
    label2id={"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
)

args_sa = TrainingArguments(
    output_dir="sa_final",
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
)

data_collator_sa = DataCollatorWithPadding(sa_tokenizer)

def compute_metrics_sa_final(eval_pred):
    preds = eval_pred.predictions.argmax(-1)
    labels = eval_pred.label_ids
    return {
        'precision': precision_score(labels, preds, average='weighted'),
        'recall': recall_score(labels, preds, average='weighted'),
        'f1': f1_score(labels, preds, average='weighted'),
        'accuracy': accuracy_score(labels, preds)
    }

trainer_sa = Trainer(
    model=model_sa,
    args=args_sa,
    train_dataset=full_train_sa_dataset,
    eval_dataset=val_sa_dataset,
    tokenizer=sa_tokenizer,
    data_collator=data_collator_sa,
    compute_metrics=compute_metrics_sa_final
)

trainer_sa.train()

# ============================
# I. Evaluate SA on Test Set
# ============================
test_reviews = df_test['review'].tolist()
test_enc_sa = tokenize_sa(test_reviews, sa_tokenizer)
if 'sentiment' in df_test.columns:
    test_labels = df_test['sentiment'].tolist()
else:
    test_labels = [0] * len(df_test)
    print("⚠️ No sentiment column in test set – using dummy labels.")

test_sa_dataset = Dataset.from_dict({
    'input_ids': test_enc_sa['input_ids'],
    'attention_mask': test_enc_sa['attention_mask'],
    'labels': test_labels
})

test_results_sa = trainer_sa.evaluate(test_sa_dataset)
print("\n=== SA Test Results ===")
print(test_results_sa)

model_sa.save_pretrained("sa_model_final")
sa_tokenizer.save_pretrained("sa_tokenizer")

# ============================
# J. Inference Example (with device fix)
# ============================
from transformers import pipeline

# NER pipeline
ner_pipeline = pipeline(
    "ner",
    model="ner_model_final",
    tokenizer="ner_tokenizer",
    aggregation_strategy="simple"
)

sample_review = "ফেস ওয়াশ টা অনেক সুন্দর, অসংখ্য ধন্যবাদ ডেলিভারি খুব দ্রুত দেয়ার জন্য।"
print("\nSample review:", sample_review)
ner_results = ner_pipeline(sample_review)
print("NER results:", ner_results)

# Sentiment inference with device handling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sa.to(device)

sample_review_sa = [sample_review]
enc = sa_tokenizer(sample_review_sa, padding=True, truncation=True, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}  # move inputs to same device

with torch.no_grad():
    logits = model_sa(**enc).logits
    pred = logits.argmax(dim=1).item()

sentiment_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
print(f"Sentiment: {sentiment_map[pred]}")

print("\nAll tasks completed successfully!")

Training samples: 3000
Test samples: 30
Train size: 2400, Validation size: 600


loading configuration file https://huggingface.co/xlm-roberta-base/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/87683eb92ea383b0475fecf99970e950a03c9ff5e51648d6eee56fb754612465.dfaaaedc7c1c475302398f09706cbb21e23951b73c6e2b3162c1c8a99bb3b62a
Model config XLMRobertaConfig {
  "_name_or_path": "xlm-roberta-base",
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.17.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}

loading file htt


=== Training NER model on full training data ===


loading configuration file https://huggingface.co/xlm-roberta-base/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/87683eb92ea383b0475fecf99970e950a03c9ff5e51648d6eee56fb754612465.dfaaaedc7c1c475302398f09706cbb21e23951b73c6e2b3162c1c8a99bb3b62a
Model config XLMRobertaConfig {
  "_name_or_path": "xlm-roberta-base",
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.17.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}

PyTorch: setting

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.086200,0.046847,0.908537,0.897590,0.903030,0.984199
2,0.024200,0.013800,0.986260,0.972892,0.979530,0.996202
3,0.011400,0.007757,0.990964,0.990964,0.990964,0.998329


...

Model weights saved in ner_model_final/pytorch_model.bin
tokenizer config file saved in ner_tokenizer/tokenizer_config.json
Special tokens file saved in ner_tokenizer/special_tokens_map.json
loading configuration file https://huggingface.co/xlm-roberta-base/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/87683eb92ea383b0475fecf99970e950a03c9ff5e51648d6eee56fb754612465.dfaaaedc7c1c475302398f09706cbb21e23951b73c6e2b3162c1c8a99bb3b62a
Model config XLMRobertaConfig {
  "_name_or_path": "xlm-roberta-base",
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": 


=== Training final SA model on full training data ===


loading configuration file https://huggingface.co/xlm-roberta-base/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/87683eb92ea383b0475fecf99970e950a03c9ff5e51648d6eee56fb754612465.dfaaaedc7c1c475302398f09706cbb21e23951b73c6e2b3162c1c8a99bb3b62a
Model config XLMRobertaConfig {
  "_name_or_path": "xlm-roberta-base",
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "NEGATIVE",
    "1": "NEUTRAL",
    "2": "POSITIVE"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "NEGATIVE": 0,
    "NEUTRAL": 1,
    "POSITIVE": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "pos

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.193900,0.002695,1.000000,1.000000,1.000000,1.000000


***** Running Evaluation *****
  Num examples = 600
  Batch size = 16
Saving model checkpoint to sa_final/checkpoint-188
Configuration saved in sa_final/checkpoint-188/config.json
Model weights saved in sa_final/checkpoint-188/pytorch_model.bin
tokenizer config file saved in sa_final/checkpoint-188/tokenizer_config.json
Special tokens file saved in sa_final/checkpoint-188/special_tokens_map.json
